# Craig Extractor Demo — Messy Text In, Accurate Quote Out

This shows the full pipeline:
1. Customer sends a messy email/message
2. Extractor pulls out product, quantity, finish, sides
3. Fuzzy matcher maps typos/synonyms to valid catalog values
4. Pricing engine returns the exact price (or escalates)

**The extractor NEVER invents a price. It only extracts specs. The pricing engine decides.**

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from extractor import extract_specs_from_text
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

def full_pipeline(message: str):
    """Run the full pipeline: extract → validate → quote."""
    print(f'CUSTOMER: "{message}"')
    print(f'{"-"*60}')
    
    # Step 1: Extract
    specs = extract_specs_from_text(message)
    
    # POA detected → escalate immediately
    if specs['poa_detected']:
        print(f'POA DETECTED: {specs["poa_detected"]}')
        print(f'CRAIG: "That\'s a custom job — let me get Justin to quote you directly."')
        print()
        return specs
    
    print(f'EXTRACTED:')
    print(f'  Product  → {specs["mapped"].get("product", "?")}')
    print(f'  Quantity → {specs["mapped"].get("quantity", "?")}')
    print(f'  Sides    → {"double" if specs["mapped"].get("double_sided") else "single" if specs["mapped"].get("double_sided") is False else "?"}')
    print(f'  Finish   → {specs["mapped"].get("finish", "?")}')
    if specs['mapped'].get('category') == 'booklet':
        print(f'  Format   → {specs["mapped"].get("format", "?")}')
        print(f'  Pages    → {specs["mapped"].get("pages", "?")}')
        print(f'  Binding  → {specs["mapped"].get("binding", "?")}')
        print(f'  Cover    → {specs["mapped"].get("cover_type", "?")}')
    print(f'  Confidence: {specs["confidence"]}')
    
    # Missing fields → Craig asks
    if specs['clarification_needed']:
        print(f'\nCRAIG WOULD ASK:')
        for q in specs['clarification_needed']:
            print(f'  → "{q}"')
    
    # Step 2: If we have a payload, hit the pricing engine
    if specs['request_payload'] and specs['endpoint']:
        print(f'\nPRICING ENGINE REQUEST:')
        print(f'  {specs["endpoint"]} → {json.dumps(specs["request_payload"])}')
        
        r = client.post(specs['endpoint'], json=specs['request_payload'])
        quote = r.json()
        
        if quote.get('success'):
            print(f'\nQUOTE RESULT:')
            print(f'  Product:     {quote["product_name"]}')
            print(f'  Base price:  EUR {quote["base_price"]}')
            if quote['surcharges_applied']:
                for s in quote['surcharges_applied']:
                    print(f'  Surcharge:   {s}')
            print(f'  Price ex VAT: EUR {quote["final_price_ex_vat"]}')
            print(f'  VAT (23%):    EUR {quote["vat_amount"]}')
            print(f'  TOTAL:        EUR {quote["total_inc_everything"]}')
            print(f'  Turnaround:   {quote["turnaround"]}')
        else:
            print(f'\nESCALATION:')
            print(f'  Reason: {quote["reason"]}')
            print(f'  {quote["message"]}')
    
    print()
    return specs

print('Pipeline ready.')

---
## Test 1 — Clean, straightforward request

In [ ]:
full_pipeline("Hi, I need 500 A5 flyers, double-sided, gloss finish.")

---
## Test 2 — Typos and casual language

In [ ]:
full_pipeline("hey can i get a price on 250 biz cards glossy both sides")

---
## Test 3 — Abbreviations and slang

In [ ]:
full_pipeline("price for 1000 dl leaflets dbl sided matt finish please")

---
## Test 4 — Soft-touch surcharge

In [ ]:
full_pipeline("how much for 250 business cards with soft touch finish")

---
## Test 5 — Large format with quantity

In [ ]:
full_pipeline("need 3 roller banners for a trade show next month")

---
## Test 6 — Large format bulk pricing kicks in

In [ ]:
full_pipeline("we need 10 foamex boards for an event")

---
## Test 7 — NCR pads with triplicate

In [ ]:
full_pipeline("hi can i get 20 A5 NCR pads 3 part please")

---
## Test 8 — Booklet with full specs

In [ ]:
full_pipeline("need 100 copies of an A5 booklet, 24 pages, saddle stitched, laminated cover")

---
## Test 9 — POA item detected → instant escalation

In [ ]:
full_pipeline("can you do z-fold brochures for us? about 500")

---
## Test 10 — Rush job detected → escalation

In [ ]:
full_pipeline("need 500 business cards ASAP can you do rush delivery")

---
## Test 11 — Missing info → Craig asks the right questions

In [ ]:
full_pipeline("how much are flyers?")

---
## Test 12 — Quantity not on sheet → pricing engine escalates

In [ ]:
full_pipeline("750 A5 flyers single sided gloss")

---
## Test 13 — Customer needs artwork

In [ ]:
full_pipeline("500 business cards gloss finish - I don't have artwork, need a design done")

---
## Test 14 — Vehicle magnetics (synonym)

In [ ]:
full_pipeline("hi how much for 2 car magnets for my van")

---
## Test 15 — Window vinyl with installation (POA flag)

In [ ]:
full_pipeline("need frosted vinyl for our office windows, about 8 sqm, including installation")